# NOVA 00 — Setup Check
Verifies GPU, HF token (secret `HF_TOKEN`), and repo access in ~1 minute.
Run this first, once.

In [1]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'
print('Bootstrap OK — repo cloned, HF token loaded.')

Cloning into '/kaggle/working/nova-ml'...


Bootstrap OK — repo cloned, HF token loaded.


In [2]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [3]:
# Verify HF token works and repos can be created under unixio
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
me = api.whoami()
print('Logged in as:', me['name'])
assert me['name'] == 'unixio', f"Expected unixio, got {me['name']}"


Logged in as: unixio


In [4]:
# Create the four model repos (idempotent)
from huggingface_hub import create_repo
for repo in ['nova-obstacle-detection', 'nova-currency-detection',
             'nova-face-detection', 'nova-face-embedding']:
    create_repo(f'unixio/{repo}', repo_type='model', exist_ok=True,
                token=os.environ['HF_TOKEN'])
    print(f'ready: https://huggingface.co/unixio/{repo}')

ready: https://huggingface.co/unixio/nova-obstacle-detection
ready: https://huggingface.co/unixio/nova-currency-detection
ready: https://huggingface.co/unixio/nova-face-detection
ready: https://huggingface.co/unixio/nova-face-embedding


In [5]:
# Smoke-test the obstacle pipeline end-to-end on COCO128 (~3 min on T4).
# Proves: ultralytics install, GPU training, TFLite INT8 export.
!pip install -q ultralytics onnx2tf onnx
!python scripts/train_obstacle.py --data coco128.yaml --epochs 1 --imgsz 320 --batch 16

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 84.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 81.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 103.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 90.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 90.3 MB/s eta 0:00:00:00:010:01
   ━━━